In [1]:
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
file_path = "../../dataStuff/UNSW_binData.csv"
data = pd.read_csv(file_path)

In [4]:
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["label"])

In [5]:
X = data.drop(columns=["label"])
y = data["label"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
model = TabNetClassifier(seed=SEED)

/usr/local/lib/python3.9/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [7]:
model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_test.values, y_test.values)],
    max_epochs=50,
    batch_size=1024,
    virtual_batch_size=128,
    patience=10
)

epoch 0  | loss: 0.10659 | val_0_auc: 0.97644 |  0:00:05s
epoch 1  | loss: 0.06158 | val_0_auc: 0.98524 |  0:00:10s
epoch 2  | loss: 0.05955 | val_0_auc: 0.98708 |  0:00:15s
epoch 3  | loss: 0.05651 | val_0_auc: 0.98419 |  0:00:20s
epoch 4  | loss: 0.05585 | val_0_auc: 0.9913  |  0:00:25s
epoch 5  | loss: 0.05526 | val_0_auc: 0.99594 |  0:00:30s
epoch 6  | loss: 0.05695 | val_0_auc: 0.9947  |  0:00:35s
epoch 7  | loss: 0.05533 | val_0_auc: 0.99597 |  0:00:40s
epoch 8  | loss: 0.05545 | val_0_auc: 0.99493 |  0:00:45s
epoch 9  | loss: 0.05432 | val_0_auc: 0.99619 |  0:00:50s
epoch 10 | loss: 0.05417 | val_0_auc: 0.99638 |  0:00:55s
epoch 11 | loss: 0.05388 | val_0_auc: 0.99527 |  0:00:59s
epoch 12 | loss: 0.05448 | val_0_auc: 0.99596 |  0:01:05s
epoch 13 | loss: 0.05321 | val_0_auc: 0.99618 |  0:01:10s
epoch 14 | loss: 0.05324 | val_0_auc: 0.99635 |  0:01:15s
epoch 15 | loss: 0.05262 | val_0_auc: 0.99628 |  0:01:20s
epoch 16 | loss: 0.05304 | val_0_auc: 0.99567 |  0:01:26s
epoch 17 | los

/usr/local/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [8]:
y_train_pred = model.predict(X_train.values)
y_val_pred = model.predict(X_test.values)

In [9]:
train_loss = model.history.history.get("loss", [None])[-1]
train_acc = accuracy_score(y_train.values, y_train_pred)

In [10]:
val_loss_key = next((key for key in model.history.history.keys() if "val" in key.lower()), None)
val_loss = model.history.history[val_loss_key][-1] if val_loss_key else None

In [11]:
val_acc = accuracy_score(y_test.values, y_val_pred)
val_prec = precision_score(y_test.values, y_val_pred, average='macro')
val_rec = recall_score(y_test.values, y_val_pred, average='macro')

In [12]:
print({
    "Train Loss": train_loss,
    "Train Accuracy": train_acc,
    "Val Loss": val_loss,
    "Val Accuracy": val_acc,
    "Val Precision": val_prec,
    "Val Recall": val_rec,
})

{'Train Loss': 0.05259999443614294, 'Train Accuracy': 0.9812282484831686, 'Val Loss': 0.9959892077347409, 'Val Accuracy': 0.9815829996920235, 'Val Precision': 0.9853019754489709, 'Val Recall': 0.9641915285429752}


In [13]:
print(f'\nFinal Accuracy: {val_acc}\n')
print(classification_report(y_test.values, y_val_pred))


Final Accuracy: 0.9815829996920235

              precision    recall  f1-score   support

           0       0.98      1.00      0.99     12337
           1       0.99      0.93      0.96      3898

    accuracy                           0.98     16235
   macro avg       0.99      0.96      0.97     16235
weighted avg       0.98      0.98      0.98     16235



# Model Prediction

In [14]:
num_features = X_train.shape[1]
sample_input = np.array([
    0.1, 20, 35, 1.5, 4.3, 10.2, 0, 1, 45, 0.67, 8, 3, 0.99, 0.5
])

In [15]:
if len(sample_input) != num_features:
    raise ValueError(f"Feature mismatch: Model expects {num_features} features, but received {len(sample_input)}")
sample_input = sample_input.reshape(1, -1)

In [16]:
predicted_label = model.predict(sample_input)
predicted_probabilities = model.predict_proba(sample_input)

In [17]:
print(f"Predicted Label: {predicted_label[0]}")
print(f"Predicted Probabilities: {predicted_probabilities}")

Predicted Label: 0
Predicted Probabilities: [[1. 0.]]
